# Stage 2 Input Preparation

This notebook creates the final Stage 2 input table by merging:
- the Stage 1 feature table (`contract_with_features_labeled.csv`) created in notebook 15_snorkel_labeling.ipynb
- the temporary Stage 2 gold label lookup table (`gold_labels_stage2_test.csv`)

Output:
- `Data/processed/contract_with_features_labeled_stage2_test.csv`

In [7]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

project_root: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-
src_path: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/src


In [8]:
import pandas as pd

from master_thesis.config import DATA_PROCESSED

In [9]:
features_path = DATA_PROCESSED / "contract_with_features_labeled.csv"
gold_path = DATA_PROCESSED / "gold_labels_stage2_test.csv"

df_features = pd.read_csv(features_path, low_memory=False)
df_gold = pd.read_csv(gold_path, low_memory=False)

print("Features table shape:", df_features.shape)
print("Gold label table shape:", df_gold.shape)

display(df_features.head())
display(df_gold.head())

Features table shape: (9201, 224)
Gold label table shape: (40, 4)


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,global_esg_yes_votes,global_esg_no_votes,global_news_yes_votes,global_news_no_votes,global_market_yes_votes,global_market_no_votes,global_supplier_macro_yes_votes,global_supplier_macro_no_votes,logistics_specific_yes_votes,logistics_specific_no_votes
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,0,0,0,0
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,0,0,0,0
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,0,0,0,0
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,0,0,0,0
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,1,0,0,0


,contract_id,contract_number,gold_y,gold_department
0,475236,475375,1,Random_Meta_Train
1,394981,395591,1,Random_Meta_Train
2,381862,382482,1,Random_Meta_Train
3,7068,7068,1,Random_Meta_Train
4,347294,348285,1,Random_Meta_Train


In [10]:
# Ensure contract_id is string-clean for a safe merge
df_features["contract_id"] = (
    df_features["contract_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

df_gold["contract_id"] = (
    df_gold["contract_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

print("Unique contracts in features:", df_features["contract_id"].nunique())
print("Unique labeled contracts in gold table:", df_gold["contract_id"].nunique())

Unique contracts in features: 2209
Unique labeled contracts in gold table: 40


In [11]:
# Merge Stage 1 feature table with Stage 2 gold labels
df_stage2_input = df_features.merge(
    df_gold,
    on="contract_id",
    how="left",
)

print("Stage 2 input shape:", df_stage2_input.shape)
print("Rows with gold_y:", df_stage2_input["gold_y"].notna().sum())
print(
    "Unique labeled contracts:",
    df_stage2_input.loc[
        df_stage2_input["gold_y"].notna(),
        "contract_id"
    ].nunique()
)

Stage 2 input shape: (9201, 227)
Rows with gold_y: 180
Unique labeled contracts: 40


In [12]:
print("Departments represented in Stage 2 labels:")
display(
    df_stage2_input[
        ["contract_id", "department", "gold_y"]
    ]
    .dropna(subset=["gold_y"])
    .drop_duplicates()
    ["department"]
    .value_counts()
)

Departments represented in Stage 2 labels:


department
Logistics                                  10
Devices & Needles                           7
Quality, Production Services & Supplies     5
Drug Product Outsourcing                    5
Raw Materials & Energy                      4
Drug Substance Outsourcing                  4
Bioprocessing and Excipients                3
Alliance, Acquisitions & PPM CoE            2
Name: count, dtype: int64

In [13]:
display(
    df_stage2_input[
        ["contract_id", "contract_number", "contract_name", "observation_year", "department", "gold_y", "gold_department"]
    ]
    .dropna(subset=["gold_y"])
    .drop_duplicates()
    .sort_values(["department", "contract_id", "observation_year"])
    .head(50)
)

KeyError: "['contract_number'] not in index"

In [14]:
out_path = DATA_PROCESSED / "contract_with_features_labeled_stage2_test.csv"
df_stage2_input.to_csv(out_path, index=False)

print("Saved:", out_path)

Saved: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/processed/contract_with_features_labeled_stage2_test.csv
